<a href="https://colab.research.google.com/github/marcospontoexe/PUC/blob/main/%5BONLINE%5D_Exemplo_1_Chatbot_baseado_em_regras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exemplo 1 - Chatbot de atendimento ao cliente baseado em regras
## Tecnólogo em Inteligência Artificial Aplicada - Agentes Conversacionais
Neste notebook iremos construir um chatbot utilizando-se das técnicas mais simples de desenvolvimento.

### Chatbot baseado em regras

É o tipo de chatbot mais simples que existe, que consiste em um dataset com respostas pré-definidas e uma série de regras que servem para encontrar estas respostas (ou intenções) no dataset.

Apesar das limitações, eles podem ser bastante eficientes e produtivos se tiverem regras bem definidas e um dataset substancial de respostas.


### Qual o contexto de nosso chatbot?

Iremos desenvolver um chatbot de interação com clientes de um restaurante. O objetivo é que o chatbot auxilie o cliente a encontrar as informações desejadas sem que precise navegar por vários links do site.

### Quais ferramentas e técnicas iremos utilizar?

*   **NLTK** - O mais famoso toolkit de Processamento de Linguagem Natural em Python.
*   **Expressões Regulares** - o pacote de regex do Python será utilizado para otimizar a busca de padrões.
*   **WordNet** - o grande banco de dados léxico e semântico, disponível em diversos idiomas.



### Construindo o chatbot
Nosso chatbot vai operar da seguinte maneira:

1.   Recebe **entrada** do usuário
2.   Procura **palavras-chave** na entrada do usuário
3.   Define a **intenção associada** à entrada
4.   Obtém a **resposta baseada na intenção** definida
5.   Mostra a **resposta** ao usuário

Portanto, temos de ter uma **tabela que associe as palavras-chave desejadas, com as intenções definidas** em nosso dataset. A seguir um exemplo:
![Exemplo de busca de intenção e resposta](https://docs.google.com/uc?export=download&id=1LnOTZiahBSv9VGrCSExbylNbxHpclMKZ)

#### Importando as bibliotecas
Vamos importar o pacote de expressões regulares do Python e também o acesso ao WordNet dado pelo NLTK.

In [ ]:
import re
import nltk
from nltk.corpus import wordnet
# Precisa efetuar o download do wordnet
nltk.download('wordnet')
# Caso use o Open Multilingual Wordnet
nltk.download('omw')

#### Construindo a lista de palavras-chave
Esta é uma etapa que pode demandar bastante tempo, principalmente de você quiser definir as palavras manualmente. Porém, quanto mais palavras-chave você tiver, maior cobertura seu chatbot vai ter.

Por questões didáticas, vamos a partir de um único termo, tentar automatizar esta busca, utilizando os sinônimos encontrados no WordNet.

In [ ]:
# Lista de palavras
palavras=['olá','horário']
# Dicionários de sinônimos
lista_sinonimos={}

# Percorre a lista de palavras
for palavra in palavras:
  sinonimos=[]
  # Busca sinônimos da palavra no wordnet em pt-br
  for syn in wordnet.synsets(palavra, lang="por"):
    # Busca formas léxicas da palavra
    for lem in syn.lemmas(lang="por"):
      # Adiciona forma na lista
      sinonimos.append(lem.name())

  # Remove palavras duplicadas e adiciona ao dicionário
  lista_sinonimos[palavra]=set(sinonimos)

print(lista_sinonimos)



> **IMPORTANTE**: Você pode aqui adicionar mais palavras manualmente ou remover algum sinônimo indesejado. Pode também adicionar frases completas ao invés de palavras-chave separadas.



In [ ]:
# Adiciona um sinônimo não existente no wordnet
lista_sinonimos['olá'].add('oi')

print(lista_sinonimos)

#### Construindo dicionário de intenções
Vamos utilizar o dicionário de sinônimos construído e mapeá-los para as possíveis intenções. Além disso, vamos formatar as palavras-chave de modo que sejam interpretadas pela ferramenta de busca de expressões regulares.

In [ ]:
# Dicionário de palavras-chave e intenções
keywords={}
keywords_dict={}

# Criando um novo registro de intenção para saudações
keywords['saudacao']=[]
# Popula a entrada criada com a lista de sinônimos da palavra-chave "olá", e formata com os metacaracteres do regex
for sin in list(lista_sinonimos['olá']):
  keywords['saudacao'].append('.*\\b'+sin+'\\b.*')

# Criando um novo registro de intenção para horário de atendimento
keywords['horario_atendimento']=[]
# Popula a entrada criada com a lista de sinônimos da palavra-chave "horário", e formata com os metacaracteres do regex
for sin in list(lista_sinonimos['horário']):
  keywords['horario_atendimento'].append('.*\\b'+sin+'\\b.*')

for intent, keys in keywords.items():
  # Une todas palavras-chave sinônimas com o operador OU
  keywords_dict[intent]=re.compile('|'.join(keys))

print(keywords_dict)

Portanto, criamos uma grande expressão regular, que pode encontrar todos termos sinônimos, para cada intenção.

O `\b` utilizado na expressão regular define que o termo entre estes boundaries que serão buscados.

O `.*` define que o regex deve buscar o padrão em todo texto de entrada.

Os sinais de `|` indicam o operador lógico OU, para que qualquer um dos sinônimos ative o gatilho de busca.



#### Definindo as respostas
Agora iremos simplesmente definir as respostas para cada intenção. Devemos sempre definir uma intenção padrão, para ser acionada caso não achemos nenhuma palavra-chave na entrada do usuário.

In [ ]:
# Define o dicionário que associa as intenções as respostas
respostas={
    'saudacao':'Olá. Como posso ajudá-lo?',
    'horario_atendimento':'Nosso horário de funcionamento é a partir das 11h30 até às 15h00.',
    'padrao':'Me desculpe, mas não entendi o que você disse. Você poderia reformular?',
}

> **NOTA**: Aqui poderíamos ainda criar mais de uma resposta para cada intenção, evitando que a interação fique repetitivo.




#### Linkando as intenções e gerando as repostas (front-end)
Aqui construíremos de fato o algoritmo que fará a interação com o usuário (o gerenciador de diálogo), onde receberemos uma entrada do mesmo, e utilizando regex, iremos buscar as palavras-chave no texto e associar com a intenção desejada.

In [ ]:
print ("Bem-vindo ao restaurante McChat. ")

# Roda indefinidamente, até que usuário peça para sair
while (True):

    # Pega o input do usuário e normaliza em letras minúsculas
    entrada = input().lower()

    # define condição de saída
    if entrada == 'sair':
        print ("Obrigado pela visita.")
        break

    matched_intent = None

    for intent,pattern in keywords_dict.items():

      # Busca as palavras-chave na entrada do usuário utilizando regex
      if re.search(pattern, entrada):

        # Se encontrou, guarda o nome da intenção correspondente
        matched_intent=intent

    # Por padrão, definimos a intenção padrão
    key='padrao'
    if matched_intent in respostas:
      key = matched_intent

    # O chatbot imprime a resposta correspondente
    print (respostas[key])

Assim, finalizamos o desenvolvimento de nosso modelo de chatbot.


### O que pode ser feito?
Como dito inicialmente, esta é a abordagem e arquitetura mais simples de chatbot, que depende muito do tamanho e cobertura da base de dados para seu sucesso.
Então, o que poderíamos fazer para melhorar?


*   Inclua mais intenções na base de dados relacionadas a dados necessários para um restaurante (e.g., endereço, cardápio, delivery, reservas)
*   Tente fazer uma regra específica para reserva de mesas no restaurante em que se salve as reservas em um dicionário e utilize regex para obter os parâmetros de data, horário e quantidade de pessoas em meio ao texto
*   Caso o restaurante abra para almoço e jantar, como resolver a intenção de horário de atendimento?



## Referências e Material complementar

* [Criando meu ChatBot com Python](https://medium.com/@erikatiliorey/criando-um-chatbot-com-python-36f24b62df6c)
* [Introducing Conversational Chat bots using Rule Based Approach](https://chatbotslife.com/introducing-conversational-chat-bots-using-rule-based-approach-c8840aeaad07)
* [Building a Simple Chatbot from Scratch in Python (using NLTK)](https://medium.com/analytics-vidhya/building-a-simple-chatbot-in-python-using-nltk-7c8c8215ac6e)
* [How to write a smart Chatbot in 40 lines without ML](https://levelup.gitconnected.com/a-chatbot-without-machine-learning-b76ad00e30c1)
* [Building a Rule-based Chatbot in Python](https://blog.datasciencedojo.com/building-a-rule-based-chatbot-in-python/)
* [Basic FAQ chat bot](https://github.com/canonicalmg/FAQ-Chat-Bot)

Este notebook foi produzido por Prof. [Lucas Oliveira](http://lattes.cnpq.br/3611246009892500).